In [2]:
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import regexp_replace
from pyspark.sql.functions import when, col, rlike

spark = SparkSession.builder.appName("SF Employment Data").getOrCreate()

In [4]:
# Read in CSV
df = spark.read.csv("Employee_Compensation.csv", header=True, inferSchema=True)

# Rename all columns to lowercase and replace spaces with underscores
df = df.select([F.col(col).alias(col.replace(' ', '_').lower()) for col in df.columns])

display(df.limit(3).toPandas())

,organization_group_code,job_family_code,job_code,year_type,year,organization_group,department_code,department,union_code,union,...,employee_identifier,salaries,overtime,other_salaries,total_salary,retirement,health_and_dental,other_benefits,total_benefits,total_compensation
0,3,1400,1404,Fiscal,2019,Human Welfare & Neighborhood Development,HSA,Human Services,790,"SEIU, Local 1021, Misc",...,37486688,60720.01,0.0,0.0,60720.01,13653.2,14733.76,4904.34,33291.30,94011.31
1,3,9700,9703,Fiscal,2019,Human Welfare & Neighborhood Development,HSA,Human Services,535,"SEIU, Local 1021, Misc",...,39646203,91677.00,0.0,0.0,91677.00,17524.2,14733.76,7411.13,39669.09,131346.09
2,3,2900,2918,Fiscal,2019,Human Welfare & Neighborhood Development,HSA,Human Services,535,"SEIU, Local 1021, Misc",...,37486043,89106.03,0.0,1540.0,90646.03,17327.2,14733.76,7401.92,39462.88,130108.91


- Check through text-based columns to ensure fields are standardized, grouped correctly.

In [5]:
df.printSchema()

root
 |-- organization_group_code: integer (nullable = true)
 |-- job_family_code: string (nullable = true)
 |-- job_code: string (nullable = true)
 |-- year_type: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- organization_group: string (nullable = true)
 |-- department_code: string (nullable = true)
 |-- department: string (nullable = true)
 |-- union_code: integer (nullable = true)
 |-- union: string (nullable = true)
 |-- job_family: string (nullable = true)
 |-- job: string (nullable = true)
 |-- employee_identifier: integer (nullable = true)
 |-- salaries: double (nullable = true)
 |-- overtime: double (nullable = true)
 |-- other_salaries: double (nullable = true)
 |-- total_salary: double (nullable = true)
 |-- retirement: double (nullable = true)
 |-- health_and_dental: double (nullable = true)
 |-- other_benefits: double (nullable = true)
 |-- total_benefits: double (nullable = true)
 |-- total_compensation: double (nullable = true)



### job_family_code

- Alphanumeric job family codes is order ; for `__UNASSIGNED__` replace with a null value

In [6]:
df = df.withColumn('job_family_code', when(col('job_family_code') == '__UNASSIGNED__', None).otherwise(col('job_family_code')))

In [7]:
job_family_code = df.select('job_family_code').distinct().toPandas()
display(job_family_code)

,job_family_code
0,2700
1,3200
2,Q000
3,1500
4,2200
5,6300
6,1100
7,8600
8,4200
9,1300


- All good

### job_code

In [8]:
job_code = df.select('job_code').distinct().toPandas()
display(job_code)

# non_numeric_job_codes = df.select('job_code').distinct().filter(~col('job_code').rlike("^[0-9]+$"))
# job_code_pd = non_numeric_job_codes.toPandas()

# job_code_pd.to_csv('job_code_pd.csv', index=False)

,job_code
0,2904
1,1436
2,8304
3,149C
4,0955
...,...
1218,4260
1219,1160
1220,5601
1221,1164


- Job code can be alphanumeric ; all good

### organization_group

In [9]:
organization_group = df.select('organization_group').distinct().toPandas()
display(organization_group)

,organization_group
0,Community Health
1,Public Protection
2,Human Welfare & Neighborhood Development
3,Culture & Recreation
4,"Public Works, Transportation & Commerce"
5,General City Responsibilities
6,General Administration & Finance


- Fields are all good, no cleaning required.

### department_code

In [10]:
department_code = df.select('department_code').distinct().toPandas()

# department_code.to_csv('department_code.csv', index=False)

df = df.withColumn('department_code', when(col('department_code') == '""', None).otherwise(col('department_code')))

- All good

### department

In [11]:
# Replace leading ALL CAPS words followed by a space with an empty string
df = df.withColumn('department', regexp_replace('department', r'^[A-Z]+ ', ''))

department = df.select('department').distinct().toPandas()
display(department)

,department
0,Fire Department
1,Mayor
2,Board Of Supervisors
3,Human Rights Commission
4,Recreation And Park Commission
...,...
76,Recreation & Park Commsn
77,Human Services Agency
78,Water Enterprise
79,Gen Svcs Agency-City Admin


- All good

### union

In [12]:
union = df.select('union').distinct().toPandas()
display(union)

,union
0,POA
1,Court-Judge
2,Unrepresented Contract Rte FBP
3,Court-Local 21 Professional
4,"Operating Engineers, Local 3"
...,...
124,Management Unrepresented Employees - MTA
125,Municipal Executive Association - Police
126,Executive Contract Employees
127,Institutional Police Officers' Association


- Standardise groups ; read in `new_union.csv` and map `union` column to standardised column `new_union`

#### Generating `new_union.csv`
- Run this cell whenever the source CSV changes to regenerate the mapping template.
- `new_union` defaults to the original value — edit the CSV to standardise any groups before running the join below.

In [ ]:
import os

raw_unions = df.select('union').distinct().toPandas().dropna().sort_values('union').reset_index(drop=True)
raw_unions['new_union'] = raw_unions['union']  # default: no change

# Only write if file does not exist — avoids overwriting manual edits
if not os.path.exists('data/new_union.csv'):
    raw_unions.to_csv('data/new_union.csv', index=False)
    print('new_union.csv created — edit new_union column to standardise groups')
else:
    print('new_union.csv already exists — delete it and re-run this cell to regenerate')

display(raw_unions)

In [14]:
new_unions_df = spark.read.csv("new_union.csv", header=True, inferSchema=True).withColumnRenamed('union', 'union_old')

df = df.join(new_unions_df, col("union") == col("union_old"), 'left').drop('union').withColumnRenamed('new_union', 'union')
display(df.toPandas())

,organization_group_code,job_family_code,job_code,year_type,year,organization_group,department_code,department,union_code,job_family,...,overtime,other_salaries,total_salary,retirement,health_and_dental,other_benefits,total_benefits,total_compensation,union_old,union
0,3,1400,1404,Fiscal,2019,Human Welfare & Neighborhood Development,HSA,Human Services,790.0,"Clerical, Secretarial & Steno",...,0.00,0.00,60720.01,13653.20,14733.76,4904.34,33291.30,94011.31,"SEIU, Local 1021, Misc","SEIU, Local 1021, Misc"
1,3,9700,9703,Fiscal,2019,Human Welfare & Neighborhood Development,HSA,Human Services,535.0,Community Development,...,0.00,0.00,91677.00,17524.20,14733.76,7411.13,39669.09,131346.09,"SEIU, Local 1021, Misc","SEIU, Local 1021, Misc"
2,3,2900,2918,Fiscal,2019,Human Welfare & Neighborhood Development,HSA,Human Services,535.0,Human Services,...,0.00,1540.00,90646.03,17327.20,14733.76,7401.92,39462.88,130108.91,"SEIU, Local 1021, Misc","SEIU, Local 1021, Misc"
3,3,2900,2918,Fiscal,2019,Human Welfare & Neighborhood Development,HSA,Human Services,535.0,Human Services,...,3355.94,337.75,89274.80,16359.16,14151.56,7096.21,37606.93,126881.73,"SEIU, Local 1021, Misc","SEIU, Local 1021, Misc"
4,3,2900,2905,Fiscal,2019,Human Welfare & Neighborhood Development,HSA,Human Services,535.0,Human Services,...,0.00,2090.00,88547.00,16925.97,14733.76,7257.89,38917.62,127464.62,"SEIU, Local 1021, Misc","SEIU, Local 1021, Misc"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
799557,2,9100,9163,Calendar,2013,"Public Works, Transportation & Commerce",MTA,Municipal Transprtn Agncy,253.0,Street Transit,...,1108.66,951.82,13202.79,2895.51,2257.40,1034.29,6187.20,19389.99,"Transport Workers - Transit Operators, Local 2...","Transport Workers - Transit Operators, Local 2..."
799558,1,H000,H002,Fiscal,2016,Public Protection,FIR,Fire Department,798.0,Fire Services,...,3522.20,18443.14,135792.38,22601.81,15861.13,2298.46,40761.40,176553.78,"Firefighters - Miscellaneous, Local 798","Firefighters - Miscellaneous, Local 798"
799559,1,H000,H002,Fiscal,2015,Public Protection,FIR,Fire Department,798.0,Fire Services,...,6164.15,17442.37,136310.26,27146.96,15230.57,2247.85,44625.38,180935.64,"Firefighters - Miscellaneous, Local 798","Firefighters - Miscellaneous, Local 798"
799560,1,H000,H002,Fiscal,2014,Public Protection,FIR,Fire Department,798.0,Fire Services,...,0.00,18592.15,131527.16,25701.56,15245.17,2161.89,43108.62,174635.78,"Firefighters - Miscellaneous, Local 798","Firefighters - Miscellaneous, Local 798"


In [15]:
union = df.select('union').distinct().toPandas()
display(union)

,union
0,POA
1,Court-Judge
2,Unrepresented Contract Rte FBP
3,Court-Local 21 Professional
4,"Operating Engineers, Local 3"
...,...
124,Management Unrepresented Employees - MTA
125,Municipal Executive Association - Police
126,Executive Contract Employees
127,Institutional Police Officers' Association


- All good!

### job

#### Generating `new_jobs.csv`
- Run this cell whenever the source CSV changes to regenerate the mapping template.
- `new_job` defaults to the original value — edit the CSV to standardise any job titles before running the join below.

In [ ]:
import os

raw_jobs = df.select('job').distinct().toPandas().dropna().sort_values('job').reset_index(drop=True)
raw_jobs['new_job'] = raw_jobs['job']  # default: no change

# Only write if file does not exist — avoids overwriting manual edits
if not os.path.exists('data/new_jobs.csv'):
    raw_jobs.to_csv('data/new_jobs.csv', index=False)
    print('new_jobs.csv created — edit new_job column to standardise titles')
else:
    print('new_jobs.csv already exists — delete it and re-run this cell to regenerate')

display(raw_jobs)

In [16]:
new_jobs_df = spark.read.csv("new_jobs.csv", header=True, inferSchema=True).withColumnRenamed('job', 'job_old')

df = df.join(new_jobs_df, col("job") == col("job_old"), 'left').drop('job_old').withColumnRenamed('new_job', 'job')
display(df.toPandas())

,organization_group_code,job_family_code,job_code,year_type,year,organization_group,department_code,department,union_code,job_family,...,other_salaries,total_salary,retirement,health_and_dental,other_benefits,total_benefits,total_compensation,union_old,union,job
0,3,1400,1404,Fiscal,2019,Human Welfare & Neighborhood Development,HSA,Human Services,790.0,"Clerical, Secretarial & Steno",...,0.00,60720.01,13653.20,14733.76,4904.34,33291.30,94011.31,"SEIU, Local 1021, Misc","SEIU, Local 1021, Misc",Clerk
1,3,9700,9703,Fiscal,2019,Human Welfare & Neighborhood Development,HSA,Human Services,535.0,Community Development,...,0.00,91677.00,17524.20,14733.76,7411.13,39669.09,131346.09,"SEIU, Local 1021, Misc","SEIU, Local 1021, Misc",HSA Emp & Training Spec II
2,3,2900,2918,Fiscal,2019,Human Welfare & Neighborhood Development,HSA,Human Services,535.0,Human Services,...,1540.00,90646.03,17327.20,14733.76,7401.92,39462.88,130108.91,"SEIU, Local 1021, Misc","SEIU, Local 1021, Misc",HSA Social Worker
3,3,2900,2918,Fiscal,2019,Human Welfare & Neighborhood Development,HSA,Human Services,535.0,Human Services,...,337.75,89274.80,16359.16,14151.56,7096.21,37606.93,126881.73,"SEIU, Local 1021, Misc","SEIU, Local 1021, Misc",HSA Social Worker
4,3,2900,2905,Fiscal,2019,Human Welfare & Neighborhood Development,HSA,Human Services,535.0,Human Services,...,2090.00,88547.00,16925.97,14733.76,7257.89,38917.62,127464.62,"SEIU, Local 1021, Misc","SEIU, Local 1021, Misc",HSA Sr Eligibility Worker
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
799557,2,9100,9163,Calendar,2013,"Public Works, Transportation & Commerce",MTA,Municipal Transprtn Agncy,253.0,Street Transit,...,951.82,13202.79,2895.51,2257.40,1034.29,6187.20,19389.99,"Transport Workers - Transit Operators, Local 2...","Transport Workers - Transit Operators, Local 2...",Transit Operator
799558,1,H000,H002,Fiscal,2016,Public Protection,FIR,Fire Department,798.0,Fire Services,...,18443.14,135792.38,22601.81,15861.13,2298.46,40761.40,176553.78,"Firefighters - Miscellaneous, Local 798","Firefighters - Miscellaneous, Local 798",Firefighter
799559,1,H000,H002,Fiscal,2015,Public Protection,FIR,Fire Department,798.0,Fire Services,...,17442.37,136310.26,27146.96,15230.57,2247.85,44625.38,180935.64,"Firefighters - Miscellaneous, Local 798","Firefighters - Miscellaneous, Local 798",Firefighter
799560,1,H000,H002,Fiscal,2014,Public Protection,FIR,Fire Department,798.0,Fire Services,...,18592.15,131527.16,25701.56,15245.17,2161.89,43108.62,174635.78,"Firefighters - Miscellaneous, Local 798","Firefighters - Miscellaneous, Local 798",Firefighter


In [17]:
df = df.toPandas().dropna()
df.isnull().sum()

organization_group_code    0
job_family_code            0
job_code                   0
year_type                  0
year                       0
organization_group         0
department_code            0
department                 0
union_code                 0
job_family                 0
job                        0
employee_identifier        0
salaries                   0
overtime                   0
other_salaries             0
total_salary               0
retirement                 0
health_and_dental          0
other_benefits             0
total_benefits             0
total_compensation         0
union_old                  0
union                      0
job                        0
dtype: int64

In [18]:
df.describe()

,organization_group_code,year,union_code,employee_identifier,salaries,overtime,other_salaries,total_salary,retirement,health_and_dental,other_benefits,total_benefits,total_compensation
count,799232.000000,799232.000000,799232.000000,7.992320e+05,799232.000000,799232.000000,799232.000000,799232.000000,799232.000000,799232.000000,799232.000000,799232.000000,799232.000000
mean,2.978133,2017.267309,492.030836,2.403244e+07,73527.794482,5928.078009,3788.295948,83244.168440,14930.992914,9929.846470,5277.613851,30138.453235,113382.621675
std,1.580805,2.712835,330.197529,2.071878e+07,49496.085886,14191.397572,7576.521464,57951.951118,11147.631447,6285.424296,3786.162569,18802.040933,75102.473245
min,1.000000,2013.000000,1.000000,0.000000e+00,-68771.780000,-22453.280000,-19131.100000,-68771.780000,-30621.430000,-3831.090000,-10636.500000,-21295.150000,-74082.610000
25%,2.000000,2015.000000,250.000000,3.396200e+04,33832.322500,0.000000,0.000000,37232.987500,5801.490000,3345.060000,2070.270000,13672.165000,51751.905000
50%,2.000000,2017.000000,535.000000,3.698544e+07,71820.020000,0.000000,766.690000,79213.415000,14976.120000,12512.550000,5168.865000,33525.010000,113066.695000
75%,4.000000,2020.000000,790.000000,4.193629e+07,105863.752500,4785.650000,4494.277500,118334.932500,21544.712500,14702.140000,7876.952500,42497.707500,161624.415000
max,7.000000,2022.000000,990.000000,5.114662e+07,651936.710000,373229.680000,342802.630000,658867.570000,160320.350000,52146.510000,35691.040000,182155.910000,807625.250000


In [19]:
print("Negative Salaries:",len(df[df['salaries']<=0]))
print("Negative Overtime:",len(df[df['overtime']<=0]))
print("Negative Other Salaries:",len(df[df['other_salaries']<=0]))
print("Negative Retirement:",len(df[df['retirement']<=0]))
print("Negative Health and Dental:",len(df[df['health_and_dental']<=0]))
print("Negative Other Benefits:",len(df[df['other_benefits']<=0]))
print("Negative Total Compensation:",len(df[df['total_compensation']<=0]))

Negative Salaries: 12875
Negative Overtime: 416223
Negative Other Salaries: 257326
Negative Retirement: 113217
Negative Health and Dental: 126393
Negative Other Benefits: 2151
Negative Total Compensation: 309


In [20]:
clean_up_df = df[['salaries','overtime','other_salaries','retirement','health_and_dental','other_benefits','total_compensation']]

for col in clean_up_df:
    df.loc[df[col] <= 0, col] = 0
    df[col]=df[col].replace(0,df[col].mean())

In [21]:
print("Negative Salaries:",len(df[df['salaries']<=0]))
print("Negative Overtime:",len(df[df['overtime']<=0]))
print("Negative Other Salaries:",len(df[df['other_salaries']<=0]))
print("Negative Retirement:",len(df[df['retirement']<=0]))
print("Negative Health and Dental:",len(df[df['health_and_dental']<=0]))
print("Negative Other Benefits:",len(df[df['other_benefits']<=0]))
print("Negative Total Compensation:",len(df[df['total_compensation']<=0]))

Negative Salaries: 0
Negative Overtime: 0
Negative Other Salaries: 0
Negative Retirement: 0
Negative Health and Dental: 0
Negative Other Benefits: 0
Negative Total Compensation: 0


In [22]:
df.to_csv('cleaned_Employee_Compensation.csv', index=False)

In [23]:
df.dtypes

organization_group_code      int32
job_family_code             object
job_code                    object
year_type                   object
year                         int32
organization_group          object
department_code             object
department                  object
union_code                 float64
job_family                  object
job                         object
employee_identifier          int32
salaries                   float64
overtime                   float64
other_salaries             float64
total_salary               float64
retirement                 float64
health_and_dental          float64
other_benefits             float64
total_benefits             float64
total_compensation         float64
union_old                   object
union                       object
job                         object
dtype: object